# Approach 3: SwissBERT on GAHD
Cells through the metrics definition prepare data and model configuration. The next section is the completed three-epoch baseline; the final section runs a separate five-epoch tuning experiment.
For tuning from a fresh kernel, run preparation through `compute_metrics`, then skip baseline training and test cells and run the five-epoch section.
Both experiments freeze embeddings and language adapters, use `de_CH`, exclude blank texts, and truncate to 128 tokens. CUDA uses FP32 because the installed X-MOD implementation fails with mixed precision.
Existing experiment files are preserved. Choose settings using GAHD dev only; Swiss German evaluation will use `gsw`.
Training correction (9 November 2026): explicit classification loss now normalizes over all accumulated examples. Transformers 5.19.0 otherwise produces fourfold gradients for these model classes with four accumulation steps. Saved outputs below predate this correction; choose new run names before retraining. Existing checkpoint scores remain measurements of the original runs.


In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

root = Path.cwd()
if not any((root / name).is_file() for name in ["data/gahd.csv", "gahd.csv"]):
    root = root.parent

gahd_path = next(root / name for name in ["data/gahd.csv", "gahd.csv"] if (root / name).is_file())
df = pd.read_csv(gahd_path)

print("\nGAHD split and label counts:")
print(pd.crosstab(df["split"], df["label"]))
print("\nMissing texts:", df["text"].isna().sum())
print("Blank texts:", df["text"].fillna("").str.strip().eq("").sum())

c:\Users\gcz12\miniconda3\envs\NLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.15 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:14:50) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.11.0+cu128
Transformers: 5.19.0
CUDA available: True
GPU: NVIDIA GeForce RTX 3080

GAHD split and label counts:
label     0     1
split            
dev     940   709
test    954   692
train  4436  3265

Missing texts: 0
Blank texts: 1


In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
from time import perf_counter

sys.path.insert(0, str(root / "src"))
from classification_loss import compute_classification_loss

MODEL = "ZurichNLP/swissbert"
SEED = 42
MAX_LEN = 128
RUN_NAME = f"swissbert_seed{SEED}"
ID2LABEL = {0: "not_hate", 1: "hate"}
LABEL2ID = {label: idx for idx, label in ID2LABEL.items()}

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
MODEL_REVISION = getattr(tokenizer, "init_kwargs", {}).get("_commit_hash")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    revision=MODEL_REVISION or "main",
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
MODEL_REVISION = getattr(model.config, "_commit_hash", None)
assert {"de_CH", "gsw"}.issubset(model.config.languages)
model.set_default_language("de_CH")
model.freeze_embeddings_and_language_adapters()

assert not any(p.requires_grad for p in model.base_model.embeddings.parameters())
assert not any(
    p.requires_grad for name, p in model.named_parameters()
    if ".adapter_modules." in name
)
assert all(p.requires_grad for p in model.classifier.parameters())

print("Model class:", type(model).__name__)
print("Available adapters:", model.config.languages)
print("Active adapter:", model.config.default_language)
print("Model revision:", MODEL_REVISION)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


Loading weights: 100%|██████████| 437/437 [00:00<00:00, 1755.02it/s]
[transformers] XmodForSequenceClassification LOAD REPORT from: ZurichNLP/swissbert
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model class: XmodForSequenceClassification
Available adapters: ['de_CH', 'fr_CH', 'it_CH', 'rm_CH', 'gsw']
Active adapter: de_CH
Model revision: None
Trainable parameters: 85646594


In [3]:
model.eval()

inputs = tokenizer(
    ["Alle Menschen verdienen gleiche Rechte."],
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LEN,
)

inputs = {key: value.to(next(model.parameters()).device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

print("Logits shape:", tuple(outputs.logits.shape))
assert outputs.logits.shape == (1, 2)

# Inspect the blank example without changing the data yet.
blank = df["text"].fillna("").str.strip().eq("")
display(df.loc[blank, ["gahd_id", "text", "label", "split"]])

Logits shape: (1, 2)


,gahd_id,text,label,split
7336,f4c7eaf2,,0,dev


In [4]:
from datasets import Dataset, DatasetDict
from transformers import DataCollatorWithPadding

# Remove empty texts, preserving all other text exactly as stored.
df_clean = df.loc[df["text"].fillna("").str.strip().ne("")].copy()

assert df_clean["label"].isin([0, 1]).all()

raw_datasets = DatasetDict({
    split: Dataset.from_pandas(
        df_clean.loc[
            df_clean["split"].eq(split),
            ["gahd_id", "text", "label"],
        ].reset_index(drop=True),
        preserve_index=False,
    )
    for split in ["train", "dev", "test"]
})

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["gahd_id", "text"],
)

# Explicit naming for the model's loss calculation.
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Pad each batch only to its longest sequence.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets)

for split, dataset in tokenized_datasets.items():
    lengths = [len(ids) for ids in dataset["input_ids"]]
    print(f"{split}: {len(dataset)} examples, longest = {max(lengths)} tokens")
    assert max(lengths) <= MAX_LEN

Map: 100%|██████████| 1646/1646 [00:00<00:00, 53874.26 examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 7701
    })
    dev: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1648
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1646
    })
})
train: 7701 examples, longest = 128 tokens
dev: 1648 examples, longest = 128 tokens
test: 1646 examples, longest = 128 tokens


In [5]:
batch = data_collator([
    tokenized_datasets["train"][i]
    for i in range(2)
])

print({key: tuple(value.shape) for key, value in batch.items()})

batch = {key: value.to(next(model.parameters()).device) for key, value in batch.items()}

model.eval()
with torch.no_grad():
    output = model(**batch)

print("Batch logits shape:", tuple(output.logits.shape))
print("Loss:", output.loss.item())

assert output.logits.shape == (2, 2)
assert torch.isfinite(output.loss)

{'labels': (2,), 'input_ids': (2, 20), 'attention_mask': (2, 20)}
Batch logits shape: (2, 2)
Loss: 0.7880224585533142


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_hate": precision_score(labels, preds, pos_label=1, zero_division=0),
        "recall_hate": recall_score(labels, preds, pos_label=1, zero_division=0),
        "f1_hate": f1_score(labels, preds, pos_label=1, zero_division=0),
        "f1_not_hate": f1_score(labels, preds, pos_label=0, zero_division=0),
        "f1_macro": f1_score(labels, preds, labels=[0, 1], average="macro", zero_division=0),
    }


In [7]:
from transformers import Trainer, TrainingArguments
from time import perf_counter
MODEL_REVISION = getattr(model.config, "_commit_hash", None)

out_dir = root / "runs" / RUN_NAME

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the GPU-enabled Python environment.")

model.set_default_language("de_CH")

# Protect an existing experiment from accidental reuse.
if out_dir.exists() and any(out_dir.iterdir()):
    raise RuntimeError(f"{out_dir} already contains files. Change RUN_NAME to use a new output directory.")

training_args = TrainingArguments(
    output_dir=str(out_dir),
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,  # effective batch size: 32
    per_device_eval_batch_size=8,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=25,
    report_to="none",
    use_cpu=False,
    # Installed X-MOD adapter routing requires matching tensor dtypes.
    fp16=False,
    bf16=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    compute_loss_func=compute_classification_loss,
)

print("Training examples:", len(tokenized_datasets["train"]))
print("Dev examples:", len(tokenized_datasets["dev"]))
print("Device:", training_args.device)
print("Model device:", next(trainer.model.parameters()).device)
print("FP16 enabled:", training_args.fp16)
print("Effective batch size:", 8 * 4)
print("Output directory:", out_dir)

Training examples: 7701
Dev examples: 1648
Device: cuda:0
Model device: cuda:0
FP16 enabled: False
Effective batch size: 32
Output directory: c:\Users\gcz12\Documents\GitHub\text-speech-processing\runs\swissbert_seed42


In [8]:
import json

start = perf_counter()
train_result = trainer.train()

print(f"Training took {(perf_counter() - start) / 60:.1f} minutes")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best dev macro-F1:", trainer.state.best_metric)

# Trainer has restored the checkpoint with the best dev macro-F1.
trainer.save_model(str(out_dir / "best_model"))
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

dev_metrics = trainer.evaluate(metric_key_prefix="dev")
trainer.save_metrics("dev", dev_metrics)

experiment = {
    "model": MODEL,
    "revision": MODEL_REVISION,
    "seed": SEED,
    "max_length": MAX_LEN,
    "training_adapter": "de_CH",
    "embeddings_and_language_adapters_frozen": True,
    "blank_texts_removed": True,
    "precision": "float32",
    "loss_normalization": "sum_cross_entropy_over_accumulated_examples",
    "device": str(trainer.args.device),
    "effective_batch_size": 32,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}

(out_dir / "experiment.json").write_text(
    json.dumps(experiment, indent=2),
    encoding="utf-8",
)

Epoch,Training Loss,Validation Loss,Accuracy,Precision Hate,Recall Hate,F1 Hate,F1 Not Hate,F1 Macro
1,2.101971,0.509112,0.740291,0.678981,0.751763,0.713521,0.762486,0.738003
2,1.773032,0.472414,0.764563,0.678135,0.861777,0.759006,0.769870,0.764438
3,1.743910,0.467643,0.779126,0.727273,0.778561,0.752044,0.800875,0.776459


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.17it/s]


Training took 6.1 minutes
Best checkpoint: c:\Users\gcz12\Documents\GitHub\text-speech-processing\runs\swissbert_seed42\checkpoint-723
Best dev macro-F1: 0.7764594351266105


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.82it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision Hate,Recall Hate,F1 Hate,F1 Not Hate,F1 Macro
1.743910,0.467643,3,0.779126,0.727273,0.778561,0.752044,0.800875,0.776459


353

In [9]:
# GAHD test is Standard German.
trainer.model.set_default_language("de_CH")

test_output = trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)

trainer.save_metrics("test", test_output.metrics)

print(pd.Series(test_output.metrics))

test_loss                    0.441457
test_accuracy                0.789793
test_precision_hate          0.731283
test_recall_hate             0.790462
test_f1_hate                 0.759722
test_f1_not_hate             0.813175
test_f1_macro                0.786449
test_runtime                 8.432100
test_samples_per_second    195.206000
test_steps_per_second       24.430000
dtype: float64


In [11]:
from sklearn.metrics import confusion_matrix

logits = torch.from_numpy(test_output.predictions)
prob_hate = torch.softmax(logits, dim=-1)[:, 1].numpy()
predictions = test_output.predictions.argmax(axis=-1)

test_predictions = raw_datasets["test"].to_pandas()
test_predictions["prediction"] = predictions
test_predictions["prob_hate"] = prob_hate

test_predictions.to_csv(
    out_dir / "gahd_test_predictions.csv",
    index=False,
)

cm = confusion_matrix(
    test_output.label_ids,
    predictions,
    labels=[0, 1],
)

display(pd.DataFrame(
    cm,
    index=["Actual not hate", "Actual hate"],
    columns=["Predicted not hate", "Predicted hate"],
))

,Predicted not hate,Predicted hate
Actual not hate,753,201
Actual hate,145,547


## Tuning experiment: five epochs
Run the two cells below after the preparation cells, or directly after the completed baseline in the current kernel.
This starts from pretrained weights, not the baseline checkpoint. Learning rate, seed, effective batch size and data stay fixed. The linear learning-rate schedule now spans five epochs.
The comparison uses dev macro-F1 only. Checkpoints save model weights without optimizer state to reduce disk use; they cannot resume the exact optimizer trajectory.


In [20]:
import gc
import json
from time import perf_counter
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed

TUNING_SEED = 44
TUNING_EPOCHS = 5
TUNING_LR = 2e-5
TUNING_RUN_NAME = "swissbert_seed44_epochs5_lr2e-5"
tuning_out_dir = root / "runs" / TUNING_RUN_NAME
baseline_dir = root / "runs" / "swissbert_seed42"
baseline_dev = json.loads((baseline_dir / "dev_results.json").read_text(encoding="utf-8"))

if not torch.cuda.is_available():
    raise RuntimeError("Select the GPU-enabled Python environment.")
if tuning_out_dir.exists() and any(tuning_out_dir.iterdir()):
    raise RuntimeError(f"{tuning_out_dir} already contains files. Change TUNING_RUN_NAME.")

# Free any prior Trainer, optimizer and model before loading fresh weights.
for variable in ["trainer", "model", "tuning_trainer", "tuning_model"]:
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()

set_seed(TUNING_SEED)
tuning_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    revision=MODEL_REVISION or "main",
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
tuning_model.set_default_language("de_CH")
tuning_model.freeze_embeddings_and_language_adapters()

# Use the same Trainer settings as the baseline; increase the training horizon.
tuning_args = TrainingArguments(
    output_dir=str(tuning_out_dir),
    num_train_epochs=TUNING_EPOCHS,
    learning_rate=TUNING_LR,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    save_only_model=True,
    logging_steps=25,
    report_to="none",
    use_cpu=False,
    fp16=False,
    bf16=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    seed=TUNING_SEED,
    data_seed=TUNING_SEED,
)
tuning_trainer = Trainer(
    model=tuning_model,
    args=tuning_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    compute_loss_func=compute_classification_loss,
)

tuning_experiment = {
    "model": MODEL,
    "revision": getattr(tuning_model.config, "_commit_hash", None) or MODEL_REVISION,
    "seed": TUNING_SEED,
    "epochs": TUNING_EPOCHS,
    "learning_rate": TUNING_LR,
    "lr_scheduler": "linear",
    "weight_decay": 0.01,
    "max_length": MAX_LEN,
    "effective_batch_size": 32,
    "training_adapter": "de_CH",
    "embeddings_and_language_adapters_frozen": True,
    "blank_texts_removed": True,
    "precision": "float32",
    "loss_normalization": "sum_cross_entropy_over_accumulated_examples",
    "device": str(tuning_args.device),
    "selection_metric": "dev_f1_macro",
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}
tuning_out_dir.mkdir(parents=True, exist_ok=True)
(tuning_out_dir / "experiment.json").write_text(json.dumps(tuning_experiment, indent=2), encoding="utf-8")
print("Device:", tuning_args.device)
print("Baseline dev macro-F1:", baseline_dev["dev_f1_macro"])
print("New run:", tuning_out_dir)


Loading weights: 100%|██████████| 437/437 [00:00<00:00, 1489.96it/s]
[transformers] XmodForSequenceClassification LOAD REPORT from: ZurichNLP/swissbert
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda:0
Baseline dev macro-F1: 0.7764594351266105
New run: c:\Users\gcz12\Documents\GitHub\text-speech-processing\runs\swissbert_seed44_epochs5_lr2e-5


In [21]:
start = perf_counter()
tuning_train_result = tuning_trainer.train()
print(f"Training took {(perf_counter() - start) / 60:.1f} minutes")
print("Best checkpoint:", tuning_trainer.state.best_model_checkpoint)
print("Best dev macro-F1:", tuning_trainer.state.best_metric)

# Trainer has restored the checkpoint selected on dev macro-F1.
tuning_trainer.save_model(str(tuning_out_dir / "best_model"))
tuning_trainer.save_metrics("train", tuning_train_result.metrics)
tuning_trainer.save_state()
tuning_dev = tuning_trainer.evaluate(metric_key_prefix="dev")
tuning_trainer.save_metrics("dev", tuning_dev)

comparison = pd.DataFrame([baseline_dev, tuning_dev], index=["baseline_3_epochs", "tuning_5_epochs"])
columns = ["dev_f1_macro", "dev_f1_hate", "dev_precision_hate", "dev_recall_hate", "dev_loss"]
display(comparison[columns].round(4))
comparison[columns].to_csv(tuning_out_dir / "dev_comparison.csv")
delta = tuning_dev["dev_f1_macro"] - baseline_dev["dev_f1_macro"]
print(f"Dev macro-F1 change: {100 * delta:+.2f} percentage points")

# Save dev predictions for error analysis; do not evaluate test while tuning.
dev_output = tuning_trainer.predict(tokenized_datasets["dev"], metric_key_prefix="dev")
dev_predictions = raw_datasets["dev"].to_pandas()
dev_predictions["prediction"] = dev_output.predictions.argmax(axis=-1)
dev_predictions["prob_hate"] = torch.softmax(torch.from_numpy(dev_output.predictions), dim=-1)[:, 1].numpy()
dev_predictions.to_csv(tuning_out_dir / "gahd_dev_predictions.csv", index=False)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Hate,Recall Hate,F1 Hate,F1 Not Hate,F1 Macro
1,2.312396,0.513703,0.739684,0.662037,0.806770,0.727273,0.751016,0.739144
2,1.837863,0.498265,0.751214,0.739200,0.651622,0.692654,0.791030,0.741842
3,1.564205,0.465197,0.782767,0.708185,0.842031,0.769330,0.794725,0.782027
4,1.405286,0.477382,0.797330,0.760779,0.771509,0.766106,0.821199,0.793653
5,1.168971,0.498131,0.797330,0.775330,0.744711,0.759712,0.824764,0.792238


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s]


Training took 10.2 minutes
Best checkpoint: c:\Users\gcz12\Documents\GitHub\text-speech-processing\runs\swissbert_seed44_epochs5_lr2e-5\checkpoint-964
Best dev macro-F1: 0.7936527930229909


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Precision Hate,Recall Hate,F1 Hate,F1 Not Hate,F1 Macro
1.168971,0.477382,5,0.797330,0.760779,0.771509,0.766106,0.821199,0.793653


,dev_f1_macro,dev_f1_hate,dev_precision_hate,dev_recall_hate,dev_loss
baseline_3_epochs,0.7765,0.7520,0.7273,0.7786,0.4676
tuning_5_epochs,0.7937,0.7661,0.7608,0.7715,0.4774


Dev macro-F1 change: +1.72 percentage points


## Final GAHD test evaluation across seeds
The configuration is fixed at five epochs and learning rate 2e-5. This section independently reloads each saved best model, evaluates German test text with `de_CH`, and saves predictions, confusion matrices and aggregate metrics. It can run from a fresh kernel without rerunning preparation or training cells.
Do not use these test scores to tune further. Swiss German evaluation will separately activate `gsw`.


In [22]:
# Final German test evaluation: reload each selected checkpoint; no training.
import gc
import json
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_hate": precision_score(labels, preds, pos_label=1, zero_division=0),
        "recall_hate": recall_score(labels, preds, pos_label=1, zero_division=0),
        "f1_hate": f1_score(labels, preds, pos_label=1, zero_division=0),
        "f1_not_hate": f1_score(labels, preds, pos_label=0, zero_division=0),
        "f1_macro": f1_score(labels, preds, labels=[0, 1], average="macro", zero_division=0),
    }


EVAL_SEEDS = [42, 43, 44]
eval_root = next(
    (candidate for candidate in [Path.cwd(), Path.cwd().parent] if (candidate / "runs").is_dir() and any((candidate / name).is_file() for name in ["data/gahd.csv", "gahd.csv"])),
    None,
)
if eval_root is None:
    raise FileNotFoundError("Run this notebook from the repository root or code directory.")
if not torch.cuda.is_available():
    raise RuntimeError("Select the GPU-enabled Python environment.")

# Release training models/optimizers when evaluating in the same kernel.
for variable in ["trainer", "model", "tuning_trainer", "tuning_model", "eval_trainer", "eval_model"]:
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()

eval_data_path = next(eval_root / name for name in ["data/gahd.csv", "gahd.csv"] if (eval_root / name).is_file())
final_test_df = pd.read_csv(eval_data_path)
final_test_df = final_test_df.loc[
    final_test_df["split"].eq("test") & final_test_df["text"].fillna("").str.strip().ne(""),
    ["gahd_id", "text", "label"],
].reset_index(drop=True)
baseline_predictions_path = eval_root / "runs" / "swissbert_seed42" / "gahd_test_predictions.csv"
if baseline_predictions_path.is_file():
    baseline_test_df = pd.read_csv(baseline_predictions_path)[["gahd_id", "text", "label"]]
    assert final_test_df.equals(baseline_test_df), "GAHD test rows differ from the baseline evaluation."
assert final_test_df["gahd_id"].is_unique
assert final_test_df["label"].isin([0, 1]).all()
print("GAHD test examples:", len(final_test_df))

final_test_rows = []
for eval_seed in EVAL_SEEDS:
    eval_run_dir = eval_root / "runs" / f"swissbert_seed{eval_seed}_epochs5_lr2e-5"
    eval_model_dir = eval_run_dir / "best_model"
    experiment_config = json.loads((eval_run_dir / "experiment.json").read_text(encoding="utf-8"))
    assert experiment_config["seed"] == eval_seed
    assert experiment_config["epochs"] == 5
    assert experiment_config["learning_rate"] == 2e-5

    eval_tokenizer = AutoTokenizer.from_pretrained(eval_model_dir, local_files_only=True)
    eval_model = AutoModelForSequenceClassification.from_pretrained(eval_model_dir, local_files_only=True)
    eval_model.set_default_language("de_CH")
    encoded_test = eval_tokenizer(
        final_test_df["text"].tolist(), truncation=True, max_length=experiment_config["max_length"]
    )
    encoded_test["labels"] = final_test_df["label"].tolist()
    eval_dataset = Dataset.from_dict(encoded_test)
    eval_args = TrainingArguments(
        output_dir=str(eval_run_dir), per_device_eval_batch_size=8,
        use_cpu=False, fp16=False, bf16=False,
        dataloader_num_workers=0, dataloader_pin_memory=True,
        report_to="none", save_strategy="no", seed=eval_seed,
    )
    eval_trainer = Trainer(
        model=eval_model, args=eval_args,
        processing_class=eval_tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=eval_tokenizer),
        compute_metrics=compute_metrics,
    )
    final_output = eval_trainer.predict(eval_dataset, metric_key_prefix="test")
    assert (final_output.label_ids == final_test_df["label"].to_numpy()).all()
    eval_trainer.save_metrics("test", final_output.metrics)

    final_predictions = final_test_df.copy()
    final_predictions["prediction"] = final_output.predictions.argmax(axis=-1)
    final_predictions["prob_hate"] = torch.softmax(torch.from_numpy(final_output.predictions), dim=-1)[:, 1].numpy()
    final_predictions.to_csv(eval_run_dir / "gahd_test_predictions.csv", index=False)
    matrix = pd.DataFrame(
        confusion_matrix(final_output.label_ids, final_predictions["prediction"], labels=[0, 1]),
        index=["Actual not hate", "Actual hate"],
        columns=["Predicted not hate", "Predicted hate"],
    )
    matrix.to_csv(eval_run_dir / "gahd_test_confusion_matrix.csv")
    final_test_rows.append({"seed": eval_seed, **final_output.metrics})
    print(f"Seed {eval_seed}: test macro-F1 = {final_output.metrics['test_f1_macro']:.4f}")
    display(matrix)

    del eval_trainer, eval_model
    gc.collect()
    torch.cuda.empty_cache()

final_test_results = pd.DataFrame(final_test_rows).set_index("seed")
final_test_columns = ["test_accuracy", "test_f1_macro", "test_f1_hate", "test_f1_not_hate", "test_precision_hate", "test_recall_hate"]
final_test_summary = final_test_results[final_test_columns].agg(["mean", "std"]).T
final_test_results.to_csv(eval_root / "runs" / "swissbert_final_gahd_test_by_seed.csv")
final_test_summary.to_csv(eval_root / "runs" / "swissbert_final_gahd_test_summary.csv")


GAHD test examples: 1646


Loading weights: 100%|██████████| 441/441 [00:00<00:00, 1646.89it/s]


Seed 42: test macro-F1 = 0.8041


,Predicted not hate,Predicted hate
Actual not hate,782,172
Actual hate,144,548


Loading weights: 100%|██████████| 441/441 [00:00<00:00, 1627.61it/s]


Seed 43: test macro-F1 = 0.8030


,Predicted not hate,Predicted hate
Actual not hate,778,176
Actual hate,142,550


Loading weights: 100%|██████████| 441/441 [00:00<00:00, 1711.41it/s]


Seed 44: test macro-F1 = 0.7925


,Predicted not hate,Predicted hate
Actual not hate,770,184
Actual hate,151,541


In [23]:
display(final_test_results[final_test_columns].round(4))
report = final_test_summary.copy()
report["percent_mean_plus_minus_sd"] = report.apply(
    lambda row: f"{100 * row['mean']:.2f} ± {100 * row['std']:.2f}", axis=1
)
display(report[["percent_mean_plus_minus_sd"]])
print("SD is the sample standard deviation across seeds, in percentage points; it is not a confidence interval.")


,test_accuracy,test_f1_macro,test_f1_hate,test_f1_not_hate,test_precision_hate,test_recall_hate
seed,,,,,,
42,0.8080,0.8041,0.7762,0.8319,0.7611,0.7919
43,0.8068,0.8030,0.7757,0.8303,0.7576,0.7948
44,0.7965,0.7925,0.7636,0.8213,0.7462,0.7818


,percent_mean_plus_minus_sd
test_accuracy,80.38 ± 0.63
test_f1_macro,79.98 ± 0.64
test_f1_hate,77.18 ± 0.72
test_f1_not_hate,82.79 ± 0.57
test_precision_hate,75.50 ± 0.78
test_recall_hate,78.95 ± 0.68


SD is the sample standard deviation across seeds, in percentage points; it is not a confidence interval.
